In [1]:
# -------------------- IMPORTS ------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import os
import time
import logging
import psycopg2
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# -------------------- LOGGING SETUP ---------------------------------------
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/postgres_ingestion.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

# -------------------- DATABASE CONFIG ---------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("Yusufrazakhan12@")  # IMPORTANT
DB_HOST = "127.0.0.1"                         # FIXED
DB_PORT = "5432"
DB_NAME = "VenderAnalysis"

# -------------------- CREATE DATABASE IF NOT EXISTS -------------------------
try:
    conn = psycopg2.connect(
        dbname="postgres",
        user=DB_USER,
        password="Yusufrazakhan12@",
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()

    cursor.execute(
        "SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,)
    )

    if not cursor.fetchone():
        cursor.execute(f'CREATE DATABASE "{DB_NAME}"')
        print(f"✅ Database '{DB_NAME}' created")
        logging.info("Database created")
        time.sleep(2)
    else:
        print(f"ℹ️ Database '{DB_NAME}' already exists")
        logging.info("Database already exists")

    cursor.close()
    conn.close()

except Exception as e:
    print("❌ Database creation failed:", e)
    logging.error(e)
    raise

# -------------------- SQLALCHEMY ENGINE ------------------------------------------
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# -------------------- INGEST FUNCTION -------------------------------------------
def ingest_db(df, table_name):
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"✅ Table ingested: {table_name}")
    logging.info(f"Table ingested: {table_name}")

# -------------------- LOAD & INGEST FILES ---------------------------------------
def load_raw_data():
    start = time.time()
    folder = "data" if os.path.exists("data") else "."

    print(f"📂 Reading files from: {folder}")
    logging.info("Ingestion started")

    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)

        try:
            if file.endswith(".csv"):
                df = pd.read_csv(file_path)
            elif file.endswith((".xlsx", ".xls")):
                df = pd.read_excel(file_path)
            elif file.endswith(".json"):
                df = pd.read_json(file_path)
            else:
                print(f"⚠️ Skipped: {file}")
                continue

            # Clean column names
            df.columns = [
                col.strip()
                .lower()
                .replace(" ", "_")
                .replace('"', "")
                .replace(";", "")[:63]
                for col in df.columns
            ]

            table_name = os.path.splitext(file)[0].lower().replace(" ", "_")
            print(f"➡️ Ingesting {file} → {table_name}")

            ingest_db(df, table_name)

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")
            logging.error(f"{file} failed: {e}")

    end = time.time()
    print(f"\n✅ Ingestion completed in {(end-start)/60:.2f} minutes")
    logging.info("Ingestion completed")

# -------------------- RUN -------------------------------------------------
if __name__ == "__main__":
    load_raw_data()

ℹ️ Database 'VenderAnalysis' already exists
📂 Reading files from: data
➡️ Ingesting begin_inventory.csv → begin_inventory
✅ Table ingested: begin_inventory
➡️ Ingesting end_inventory.csv → end_inventory
✅ Table ingested: end_inventory
➡️ Ingesting purchases.csv → purchases
✅ Table ingested: purchases
➡️ Ingesting purchase_prices.csv → purchase_prices
✅ Table ingested: purchase_prices
➡️ Ingesting sales.csv → sales
✅ Table ingested: sales
➡️ Ingesting vendor_invoice.csv → vendor_invoice
✅ Table ingested: vendor_invoice

✅ Ingestion completed in 215.07 minutes


In [6]:
# import library
import pandas as pd
import psycopg2

In [3]:
# cerate a connection 
conn = psycopg2.connect(
    host="localhost",
    database="VenderAnalysis",   
    user="postgres",       
    password="Yusufrazakhan12@", 
    port="5432"
)

In [4]:
cur = conn.cursor()

# Table names fetch
cur.execute("""
    SELECT table_name 
    FROM information_schema.tables
    WHERE table_schema = 'public';
""")

tables = cur.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(table[0])

Tables in database:
vendor_sales_summary
begin_inventory
end_inventory
purchases
purchase_prices
sales
vendor_invoice


In [7]:
# fetch table 
data = pd.read_sql('SELECT * FROM "vendor_sales_summary";', conn)
data.head(2)

C:\Users\mohdy\AppData\Local\Temp\ipykernel_26972\234826891.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql('SELECT * FROM "vendor_sales_summary";', conn)


,vendornumber,vendorname,brand,description,purchaseprice,actualprice,volume,totalpurchasequantity,totalpurchasedollars,totalsalesquantity,totalsalesdollars,totalsalesprice,totalexcisetax,freightcost,grossprofit,profitmargin,stockturnover,salestopurchaseratio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.60,142049,5101919.51,672819.31,260999.20,68601.68,1290667.91,25.30,1.0,1.34
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.22,160247,4819073.49,561512.37,294438.66,144929.24,1015032.27,21.06,1.0,1.27
